# Google Play Store - Analisi Strategica

## Obiettivo del Progetto
Questo notebook analizza il mercato delle applicazioni Android per identificare nicchie profittevoli e categorie ad alto potenziale ("Opportunity"). L'analisi supporta decisioni strategiche per il lancio di nuovi prodotti digitali (MVP).

## Struttura dell'Analisi
1. **Data Cleaning**: Pulizia e normalizzazione di dataset grezzi (parsing di prezzi, installazioni, rating).
2. **Quality Check**: Analisi di valori mancanti e outlier.
3. **Analisi Esplorativa (EDA)**: Distribuzioni di rating, installazioni e correlazioni.
4. **Analisi Strategica**: Calcolo dell'Opportunity Score (domanda vs offerta).
5. **Forecast**: Stima dei ricavi potenziali per le categorie top.
6. **Reporting**: Generazione automatica di un PDF riassuntivo.

## Note Metodologiche
* **Correlazione vs Causalità**: Nelle sezioni seguenti verranno mostrate matrici di correlazione (es. Rating vs Installazioni). È fondamentale ricordare che *la correlazione non implica causalità*. Un alto rating potrebbe non *causare* più installazioni; potrebbe esserci una variabile latente (es. budget marketing, qualità del brand) che influenza entrambe.

In [ ]:
# SEZIONE 1: Import e Configurazione
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
from scipy import stats

# Librerie per PDF
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import cm

# Configurazione grafica
sns.set(style='whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0})

# Setup Cartelle
OUT_DIR = 'googleplay_analysis_outputs'
PDF_PATH = 'googleplay_report_full.pdf'
os.makedirs(OUT_DIR, exist_ok=True)

print("Setup completato. Directory output:", OUT_DIR)

# SEZIONE 2: Caricamento Dati (Cloud)
# Uso un URL raw di GitHub per rendere il notebook eseguibile ovunque
DATA_URL = 'https://raw.githubusercontent.com/gauthamp10/Google-Play-Store-Data-Analysis/master/googleplaystore.csv'

print(f"Scaricamento dati da: {DATA_URL} ...")
try:
    response = requests.get(DATA_URL)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    print(f"Download completato. Righe lette: {len(df)}")
except Exception as e:
    print(f"ERRORE nel download: {e}")
    # Fallback: decommenta la riga sotto se hai il file locale
    # df = pd.read_csv('googleplaystore.csv')

# SEZIONE 3: Pulizia Dati (Cleaning)
# Pulizia nomi colonne
df.columns = [c.strip() for c in df.columns]

# Rimozione duplicati
df = df.drop_duplicates(subset=['App', 'Category'], keep='first')

# Funzioni di parsing robuste
def parse_installs(x):
    try:
        if pd.isna(x): return np.nan
        s = str(x).strip().replace(',', '').replace('+', '')
        # Controllo se finisce con cifra (evita righe header o "Free")
        if not s[-1].isdigit(): return np.nan
        return int(s)
    except:
        return np.nan

def parse_price(x):
    try:
        if pd.isna(x): return 0.0
        s = str(x).strip().replace('$', '')
        if s.lower() == 'everyone': return 0.0
        return float(s)
    except:
        return 0.0

# Applicazione parsing
df['Installs'] = df['Installs'].apply(parse_installs)
df['Price'] = df['Price'].apply(parse_price)
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce').fillna(0).astype(int)

# Creazione colonna Type e Log Installazioni
df['Type'] = df['Price'].apply(lambda p: 'Paid' if p > 0 else 'Free')
df['ln_Installs'] = np.log1p(df['Installs'])

# Rimozione righe con categoria errata (es. '1.9')
df = df[df['Category'] != '1.9']

print("Pulizia completata. Shape finale:", df.shape)
df.head(3)

In [ ]:
# SEZIONE 4: Data Quality Check & Outlier Analysis

print("--- ANALISI QUALITÀ DATI ---")

# 1. Valori Mancanti
missing = df.isnull().sum()
print("\nValori mancanti (Top 5):")
print(missing[missing > 0].head())

# 2. Gestione Rating Mancanti
# Qui li mantengo come NaN ma li segnalo
print(f"App senza rating: {df['Rating'].isna().sum()}")

# 3. Analisi Outlier (Grafica)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot Rating
sns.boxplot(x=df['Rating'], ax=axes[0], color='orange')
axes[0].set_title('Distribuzione Rating (Boxplot)')
axes[0].set_xlabel('Rating (1-5)')

# Boxplot Prezzi (Log scale per vedere meglio)
# Vengono filtrati i prezzi > 0 per il grafico
paid_apps = df[df['Price'] > 0]
if not paid_apps.empty:
    sns.boxplot(x=paid_apps['Price'], ax=axes[1], color='red')
    axes[1].set_title('Distribuzione Prezzi (App a pagamento)')
    axes[1].set_xscale('log') # Scala logaritmica per gestire outlier estremi
    axes[1].set_xlabel('Prezzo ($) - Scala Log')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'data_quality_outliers.png'))
plt.show()

print("Nota: I valori estremi nei prezzi (es. 400$) sono spesso app 'status symbol' o errori.")

In [ ]:
# SEZIONE 5: Analisi Strategica e Visualizzazioni

# A. Violin Plot (Rating per Top Categorie)
# Vengono selezionate le 5 categorie più numerose
top_cats = df['Category'].value_counts().nlargest(5).index
df_top = df[df['Category'].isin(top_cats)]

plt.figure(figsize=(12, 6))
sns.violinplot(x='Category', y='Rating', data=df_top, palette='viridis')
plt.title('Densità dei Rating nelle Top 5 Categorie')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'rating_violin_plot.png'))
plt.show()

# B. Calcolo Opportunity Score
# Raggruppiamo per categoria
cat_stats = df.groupby('Category').agg(
    AppCount=('App', 'count'),
    MedianInstalls=('Installs', 'median'),
    TotalInstalls=('Installs', 'sum'),
    AvgRating=('Rating', 'mean')
).reset_index()

# Calcolo Ranking
# AppCount alto = Alta competizione -> Rank invertito o usato al denominatore
# MedianInstalls alto = Alta domanda

# Formula Opportunity: (Domanda / Competizione) normalizzata
cat_stats['opportunity_score'] = cat_stats['MedianInstalls'] / (cat_stats['AppCount'] + 1)
# Normalizziamo con log per visualizzazione migliore
cat_stats['norm_score'] = np.log1p(cat_stats['opportunity_score'])

# C. Opportunity Map (Scatter Plot)
plt.figure(figsize=(10, 8))
scatter = sns.scatterplot(
    data=cat_stats,
    x='AppCount',
    y='MedianInstalls',
    size='TotalInstalls',
    hue='norm_score',
    palette='RdYlGn', # Rosso = Bassa opportunità, Verde = Alta
    sizes=(50, 600),
    alpha=0.7
)

# Setup assi logaritmici per vedere meglio i dati distribuiti
plt.xscale('log')
plt.yscale('log')
plt.title('Opportunity Map: Competizione vs Domanda')
plt.xlabel('Competizione (Numero App) - Log')
plt.ylabel('Domanda (Installazioni Mediane) - Log')

# Etichette per le categorie migliori (Top 5 per score)
top_opps = cat_stats.nlargest(5, 'opportunity_score')
for i in range(len(top_opps)):
    plt.text(
        top_opps.iloc[i]['AppCount'],
        top_opps.iloc[i]['MedianInstalls'],
        top_opps.iloc[i]['Category'],
        horizontalalignment='right',
        weight='bold',
        color='black'
    )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'opportunity_map.png'))
plt.show()

# Salvataggio CSV dati
cat_stats.sort_values('opportunity_score', ascending=False).to_csv(os.path.join(OUT_DIR, 'category_analysis.csv'), index=False)
print("Analisi completata. Grafici salvati in:", OUT_DIR)

In [ ]:
# SEZIONE 6: Generazione PDF Report

def create_pdf():
    doc = SimpleDocTemplate(PDF_PATH, pagesize=A4)
    story = []
    styles = getSampleStyleSheet()
    
    # 1. Titolo e Intro
    title = Paragraph("Report Strategico: Google Play Store", styles['Title'])
    story.append(title)
    story.append(Spacer(1, 12))
    
    intro_text = f"""
    <b>Data Analisi:</b> {pd.Timestamp.now().strftime('%Y-%m-%d')}<br/>
    <b>Totale App:</b> {len(df)}<br/>
    <b>Obiettivo:</b> Identificare categorie con alta domanda e bassa competizione (Blue Ocean).
    """
    story.append(Paragraph(intro_text, styles['Normal']))
    story.append(Spacer(1, 12))
    
    # 2. Inserimento Immagini generate
    # Lista immagini da cercare
    images = [
        ('Opportunity Map (Strategia)', 'opportunity_map.png'),
        ('Analisi Qualità Dati (Outliers)', 'data_quality_outliers.png'),
        ('Distribuzione Rating (Violin)', 'rating_violin_plot.png')
    ]
    
    for title_img, filename in images:
        path = os.path.join(OUT_DIR, filename)
        if os.path.exists(path):
            story.append(Paragraph(title_img, styles['Heading2']))
            # Si ridimensiona l'immagine per stare nel PDF (larghezza 16cm)
            img = Image(path, width=16*cm, height=9*cm)
            story.append(img)
            story.append(Spacer(1, 12))
    
    story.append(PageBreak())
    
    # 3. Tabella Top Categorie
    story.append(Paragraph("Top 10 Categorie per Opportunità", styles['Heading2']))
    
    # Prepariamo i dati
    top_df = cat_stats.sort_values('opportunity_score', ascending=False).head(10)
    top_df = top_df[['Category', 'AppCount', 'MedianInstalls', 'AvgRating']]
    # Arrotondiamo
    top_df['AvgRating'] = top_df['AvgRating'].round(2)
    
    # Header + Dati
    data = [top_df.columns.tolist()] + top_df.values.tolist()
    
    # Creazione Tabella con larghezze fisse per evitare overflow
    # A4 larghezza utile è circa 19cm. Usate: 7cm, 3cm, 5cm, 3cm
    table = Table(data, colWidths=[7*cm, 3*cm, 5*cm, 3*cm])
    
    # Stile Tabella
    style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.darkblue), # Header blu scuro
        ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke), # Testo header bianco
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,0), 10),
        ('BOTTOMPADDING', (0,0), (-1,0), 12),
        ('BACKGROUND', (0,1), (-1,-1), colors.beige), # Righe dati beige
        ('GRID', (0,0), (-1,-1), 1, colors.black)
    ])
    table.setStyle(style)
    story.append(table)
    
    # 4. Conclusioni
    story.append(Spacer(1, 24))
    conclusions = """
    <b>Conclusioni Strategiche:</b><br/>
    Le categorie visualizzate in alto nella tabella mostrano un alto numero di installazioni mediane 
    rispetto al numero di competitor. Si consiglia di approfondire le prime 3 categorie per lo sviluppo di un MVP.
    """
    story.append(Paragraph(conclusions, styles['Normal']))
    
    # Build
    try:
        doc.build(story)
        print(f"PDF creato con successo: {PDF_PATH}")
    except Exception as e:
        print(f"Errore generazione PDF: {e}")

# Esecuzione
create_pdf()